# MindCare: Mental Health Risk Classification Pipeline
This notebook details the end-to-end training, hyperparameter tuning, and evaluation for MindCare's 7-class mental health risk screening model.

### Target Classes (7 Categories):
* `Normal`, `Depression`, `Suicidal`, `Anxiety`, `Bipolar`, `Stress`, `Personality disorder`

### Dataset:
* `research/datasets/raw/Combined_Data.csv` (51,017 cleaned records)


In [ ]:
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, recall_score, make_scorer


## 1. Data Ingestion & Preprocessing
Load the raw benchmark dataset, rename schema columns, and clean text strings.


In [ ]:
DATA_PATH = "../datasets/raw/Combined_Data.csv"

df = pd.read_csv(DATA_PATH)
df = df.rename(columns={"statement": "text", "status": "label"})
df = df[["text", "label"]].dropna(subset=["text", "label"])

print(f"Loaded raw records: {len(df)}")
df.head()


In [ ]:
def clean_text(text: str) -> str:
    """Standard text preprocessor: removes URLs, handles lowercasing, and strips extra spaces."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s!?,.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)
df = df.drop_duplicates(subset=['text'])
df = df[df['text'].str.len() > 0]

print(f"Cleaned unique records: {len(df)}")
print("\nClass Distribution:")
print(df['label'].value_counts())


## 2. Stratified Train / Test Split
Split into 80% training and 20% testing partitions while preserving relative class distributions.


In [ ]:
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['label']
)

print(f"Train split: {len(train_df)} | Test split: {len(test_df)}")


## 3. Sublinear TF-IDF Feature Extraction
Transform textual statements into n-gram TF-IDF representations (1-2 word combinations).


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df['text'])
X_test = vectorizer.transform(test_df['text'])
y_train = train_df['label']
y_test = test_df['label']
labels = sorted(y_train.unique().tolist())

print(f"TF-IDF Matrix shape (Train): {X_train.shape}")


## 4. Model Training & Comparison
We compare `LogisticRegression` (with balanced class weighting) and `LinearSVC`.


In [ ]:
# Model 1: Balanced Logistic Regression
print("Training Balanced Logistic Regression...")
logreg = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0, random_state=42)
logreg.fit(X_train, y_train)

# Model 2: Linear Support Vector Classifier
print("Training LinearSVC...")
svc = LinearSVC(class_weight='balanced', random_state=42)
svc.fit(X_train, y_train)


## 5. Performance Evaluation
Generate classification reports and confusion matrices for both models.


In [ ]:
logreg_preds = logreg.predict(X_test)
print("=== Logistic Regression Classification Report ===")
print(classification_report(y_test, logreg_preds, target_names=labels))

svc_preds = svc.predict(X_test)
print("=== LinearSVC Classification Report ===")
print(classification_report(y_test, svc_preds, target_names=labels))


In [ ]:
# Confusion Matrix for Logistic Regression
cm = confusion_matrix(y_test, logreg_preds, labels=labels)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title("Mental Health Risk Classification: Confusion Matrix")
plt.xlabel("Predicted Risk")
plt.ylabel("True Risk")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 6. Targeted Hyperparameter Tuning (Suicidal Class Recall Scorer)
In mental health screening, maximizing recall on the `Suicidal` class is crucial for patient safety.


In [ ]:
def suicidal_recall_scorer(y_true, y_pred):
    return recall_score(y_true, y_pred, labels=['Suicidal'], average=None)[0]

param_grid = {'C': [0.5, 1.0, 2.0]}
grid = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    param_grid,
    scoring=make_scorer(suicidal_recall_scorer),
    cv=3,
    n_jobs=-1
)
grid.fit(X_train, y_train)

print(f"Optimal Regularization C: {grid.best_params_}")
tuned_preds = grid.best_estimator_.predict(X_test)
print(classification_report(y_test, tuned_preds, target_names=labels))


## 7. Model Serialization Export Demonstration
*Note: To avoid unintended production side effects, models can be saved to a local demo folder.*


In [ ]:
# Export demonstration
import os
os.makedirs("exports", exist_ok=True)

with open("exports/demo_mh_model.pkl", "wb") as f:
    pickle.dump(logreg, f)

with open("exports/demo_mh_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Export completed successfully.")
